
# Семинар: работа с таблицами, CSV, Excel, xlwings и OLAP

Ноутбук основан на материалах лекции **«Работа с таблицами»**: хранение табличных данных, CSV, Excel в Python, `openpyxl`, `pandas`, `xlwings`, формулы, диапазоны, графики и основы OLTP/OLAP.

## Правила
1. В начале ноутбука задаётся `STUDENT_ID`.
2. По `STUDENT_ID` генерируются **индивидуальные исходные данные** и локальные файлы.
3. Во многих заданиях используются:
   - контрольные суммы;
   - сквозные зависимости между предыдущими и следующими задачами;
   - локально созданные CSV/XLSX-артефакты;
   - персонализированные численные результаты.

4. Порядок выполнения
- Не заменяй вычисления ручными константами.
- Все ответы сохраняй в переменные с указанными именами.
- Если задание требует файл, реально создай его на диске.
- Для текстовых выводов обязательно опирайся на вычисленные числа.
- Для заданий 13–16 достаточно подготовить корректный код/текст, выполнять `xlwings` не требуется.

## Структура
- Первая пара: задания 1–10.
- Вторая пара: задания 11–20.

## Представляемые результаты
1. Заполненный ноутбук.
2. Все созданные локальные файлы:
   - `employees_export.csv`
   - `analytics_pack.xlsx`
   - `final_submission.xlsx`
   - каталог `star_schema/`
3. Краткие комментарии там, где они требуются условием.

## Источники
- Лекция «Работа с таблицами».
- `csv` в Python: `https://docs.python.org/3/library/csv.html`
- `pandas.read_csv`: `https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html`
- `openpyxl`: `https://openpyxl.readthedocs.io/`
- `xlwings`: `https://docs.xlwings.org/`


In [ ]:
from pathlib import Path
import json
import csv
import hashlib
import random
from collections import defaultdict
from datetime import date, datetime, timedelta

import numpy as np
import pandas as pd
from openpyxl import Workbook, load_workbook
from openpyxl.styles import PatternFill, Font, Alignment
from openpyxl.chart import BarChart, LineChart, Reference
from openpyxl.workbook.defined_name import DefinedName

In [ ]:
# Впишите свой идентификатор свой идентификатор.
STUDENT_ID = "123456"  # например: "123456" из "123456@edu.fa.ru"
SEED = int(hashlib.sha256(STUDENT_ID.encode("utf-8")).hexdigest()[:8], 16)

rng = random.Random(SEED)
np_rng = np.random.default_rng(SEED)

BASE_DIR = Path("data")
BASE_DIR.mkdir(exist_ok=True)

def stable_token(obj) -> str:
    payload = json.dumps(obj, ensure_ascii=False, sort_keys=True, default=str)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()[:12]

In [ ]:
# ------------------------------
# Генерация индивидуальных файлов
# ------------------------------
def prepare_variant_files(base_dir: Path, seed: int):
    rng = random.Random(seed)
    np_rng = np.random.default_rng(seed)

    # ---------- 1. Hierarchical JSON ----------
    first_names = ["Анна", "Борис", "Вера", "Глеб", "Дарья", "Егор", "Жанна", "Илья"]
    last_names = ["Соколова", "Иванов", "Миронов", "Петрова", "Федоров", "Климова"]
    regions = ["north", "south", "west", "east"]
    industries = ["fintech", "edtech", "retail", "logistics"]
    orgs = ["Orion Labs", "Delta Soft", "Astra Data", "Vector BI", "Nova Tech"]
    schools = ["МГУ", "СПбГУ", "МФТИ", "ВШЭ", "ИТМО"]
    cities = ["Москва", "Казань", "Самара", "Томск", "Пермь"]
    users = []
    for uid in range(1, 6):
        positions = []
        for _ in range(rng.randint(1, 3)):
            positions.append({
                "job_title": rng.choice(["analyst", "engineer", "team lead", "researcher"]),
                "organization": rng.choice(orgs),
                "years": rng.randint(1, 5)
            })
        education = []
        for _ in range(rng.randint(1, 2)):
            start = rng.randint(2012, 2018)
            finish = start + rng.randint(2, 4)
            education.append({
                "school_name": rng.choice(schools),
                "start": start,
                "end": finish
            })
        contact_keys = ["email", "telegram", "blog", "city"]
        rng.shuffle(contact_keys)
        contact_info = {}
        for key in contact_keys[:rng.randint(2, 4)]:
            if key == "email":
                contact_info[key] = f"user{uid}_{seed % 1000}@example.org"
            elif key == "telegram":
                contact_info[key] = f"@user_{uid}_{seed % 97}"
            elif key == "blog":
                contact_info[key] = f"https://example.org/blog/{uid}/{seed % 13}"
            elif key == "city":
                contact_info[key] = rng.choice(cities)
        users.append({
            "user_id": 100 + uid,
            "first_name": rng.choice(first_names),
            "last_name": rng.choice(last_names),
            "region_id": rng.choice(regions),
            "industry_id": rng.choice(industries),
            "summary": f"Пользователь варианта {uid} с seed={seed}",
            "positions": positions,
            "education": education,
            "contact_info": contact_info
        })
    profiles_path = base_dir / "profiles_variant.json"
    with profiles_path.open("w", encoding="utf-8") as f:
        json.dump(users, f, ensure_ascii=False, indent=2)

    # ---------- 2. Messy CSV ----------
    inv_rows = [
        ["SKU001", 'Сыр "Пармезан"', "food", round(100 + rng.random() * 300, 2), rng.randint(5, 50),
         "Партия с пометкой: \"проверить\""],
        ["SKU002", "Кофе арабика, 1 кг", "food", round(300 + rng.random() * 500, 2), rng.randint(5, 50),
         "Требуется анализ аромата\nи повторная дегустация"],
        ["SKU003", "USB-C Dock", "electronics", round(1000 + rng.random() * 3000, 2), rng.randint(5, 50),
         "Совместим с Linux, macOS, Windows"],
        ["SKU004", "Монитор 27\"", "electronics", round(7000 + rng.random() * 8000, 2), rng.randint(5, 50),
         "Есть 2 битых пикселя; вернуть поставщику"],
        ["SKU005", "Тетрадь A4", "office", round(50 + rng.random() * 100, 2), rng.randint(20, 100),
         "Обычная поставка"],
        ["SKU006", "Маркер, синий", "office", round(20 + rng.random() * 50, 2), rng.randint(20, 100),
         "Внутри коробки найден лист\nс ручными правками"],
        ["SKU007", "SSD 1TB", "electronics", round(4000 + rng.random() * 5000, 2), rng.randint(5, 40),
         "Новая ревизия контроллера"],
        ["SKU008", "Чай зелёный", "food", round(80 + rng.random() * 150, 2), rng.randint(5, 60),
         "Комментарий, содержащий, много, запятых"],
    ]
    messy_path = base_dir / "messy_inventory.csv"
    with messy_path.open("w", encoding="utf-8", newline="") as f:
        writer = csv.writer(f, delimiter=";", quotechar='"', quoting=csv.QUOTE_MINIMAL)
        writer.writerow(["sku", "name", "category", "price", "qty", "comment"])
        writer.writerows(inv_rows)

    # ---------- 3. Ragged CSV ----------
    ragged_path = base_dir / "ragged_people.csv"
    ragged_lines = [
        "surname;name;role;city",
        "Иванов;Павел;analyst;Москва",
        "Соколова;Анна;engineer",
        "Петров;Глеб;manager;Казань;EXTRA_NOTE",
        "Климова;Дарья;;;EXTRA1;EXTRA2",
        "Федоров;Илья;intern;Пермь",
    ]
    ragged_path.write_text("\n".join(ragged_lines), encoding="utf-8")

    # ---------- 4. Employees CSV ----------
    departments = ["analytics", "backend", "ml", "finance"]
    employees = []
    base_date = date(2020, 1, 1)
    for idx in range(1, 13):
        hired = base_date + timedelta(days=rng.randint(0, 1500))
        employees.append({
            "employee_id": f"E{idx:03d}",
            "full_name": f"Сотрудник_{idx}_{seed % 100}",
            "department": rng.choice(departments),
            "hired": hired.strftime("%d.%m.%Y"),
            "salary": int(rng.randint(70_000, 180_000) / 1000) * 1000,
            "sick_days": rng.randint(0, 12),
            "bonus_rate": round(rng.uniform(0.03, 0.20), 3),
        })
    employees_path = base_dir / "employees_variant.csv"
    with employees_path.open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=["employee_id", "full_name", "department", "hired", "salary", "sick_days", "bonus_rate"],
            delimiter=";"
        )
        writer.writeheader()
        writer.writerows(employees)

    # ---------- 5. OLTP transactions ----------
    regions = ["RU-C", "RU-NW", "RU-S", "RU-U"]
    product_groups = ["hardware", "software", "service"]
    products = {
        "hardware": ["laptop", "ssd", "router"],
        "software": ["crm", "erp", "etl"],
        "service": ["audit", "support", "training"]
    }
    customers = [f"CUST_{i:02d}" for i in range(1, 11)]
    tx_rows = []
    start_dt = date(2024, 1, 1)
    for tx_id in range(1, 181):
        group = rng.choice(product_groups)
        product = rng.choice(products[group])
        tx_date = start_dt + timedelta(days=rng.randint(0, 150))
        units = rng.randint(1, 9)
        price = {
            "hardware": rng.randint(20_000, 80_000),
            "software": rng.randint(8_000, 30_000),
            "service": rng.randint(5_000, 25_000)
        }[group]
        if group == "software":
            units = rng.randint(1, 20)
        discount = rng.choice([0.0, 0.0, 0.03, 0.05, 0.10])
        tx_rows.append({
            "tx_id": tx_id,
            "tx_date": tx_date.isoformat(),
            "region": rng.choice(regions),
            "customer_id": rng.choice(customers),
            "product_group": group,
            "product": product,
            "units": units,
            "unit_price": price,
            "discount": discount,
        })
    tx_path = base_dir / "transactions_oltp.csv"
    pd.DataFrame(tx_rows).to_csv(tx_path, index=False)

    # ---------- 6. Statements for OLTP/OLAP ----------
    statements = pd.DataFrame({
        "statement_id": range(1, 9),
        "statement": [
            "Много клиентов одновременно вносят небольшие изменения в данные.",
            "Запросы нерегламентированные, с большим числом группировок и агрегатов.",
            "Данные редко обновляются и приходят пакетной загрузкой.",
            "Требуется минимальное время отклика на транзакцию.",
            "Важен анализ временных зависимостей по месяцам и кварталам.",
            "Схема стремится к нормализации и отсутствию дублирования.",
            "Пользователи — аналитики и менеджеры, а не операторы.",
            "Транзакции обычно затрагивают небольшой объём данных."
        ]
    })
    statements_path = base_dir / "oltp_olap_statements.csv"
    statements.to_csv(statements_path, index=False)

    return {
        "profiles": profiles_path,
        "messy_inventory": messy_path,
        "ragged_people": ragged_path,
        "employees": employees_path,
        "transactions": tx_path,
        "statements": statements_path,
    }

PATHS = prepare_variant_files(BASE_DIR, SEED)
print("STUDENT_ID =", STUDENT_ID)
print("SEED =", SEED)
print("Файлы варианта:")
for name, path in PATHS.items():
    print(f"  {name}: {path}")

## Задание 1. Нормализация иерархического JSON

Прочитай файл `profiles_variant.json` и построй три таблицы:

- `users_df`: `user_id, first_name, last_name, region_id, industry_id, summary`
- `positions_df`: `user_id, position_idx, job_title, organization, years`
- `education_df`: `user_id, education_idx, school_name, start, end`

Требования:
1. Индексы `position_idx` и `education_idx` должны начинаться с 1 внутри каждого пользователя.
2. Таблицы должны быть отсортированы по `user_id`, затем по индексу вложенного элемента.
3. Сохрани результат в словарь `task01_result = {"users": users_df, "positions": positions_df, "education": education_df}`.
        

In [ ]:
# YOUR CODE HERE

## Задание 2. Динамическая схема документа

Из того же JSON сформируй таблицу `contacts_df` со столбцами:

`user_id, email, telegram, blog, city`

Требования:
1. Если поля нет, используй `None`.
2. Не меняй порядок строк: сортировка по `user_id`.
3. Вычисли контрольную сумму  
   `task02_checksum = stable_token(contacts_df.to_dict(orient="records"))`.
        

In [ ]:
# YOUR CODE HERE

## Задание 3. Сложный CSV с разделителем, кавычками и переводами строк

Используя модуль `csv`, корректно прочитай файл `messy_inventory.csv`.

Нужно получить:
1. `task03_multiline_count` — число строк, где в поле `comment` есть перевод строки.
2. `task03_category_stats` — словарь со средней ценой по категориям.
3. `task03_top3` — список из трёх `sku` с самыми длинными комментариями.

Все вычисления сделать **через `csv.reader`**, а не через `pandas.read_csv`.
        

In [ ]:
# YOUR CODE HERE

## Задание 4. DictReader, restkey и restval

Прочитай `ragged_people.csv` с помощью `csv.DictReader`.

Требования:
1. Используй разделитель `;`.
2. Передай явный список `fieldnames = ["surname", "name", "role", "city"]`.
3. Используй `restkey="overflow"` и `restval="MISSING"`.
4. Исключи первую строку-фактический заголовок вручную.
5. Построй `task04_df` с колонками:
   - `surname`
   - `name`
   - `role`
   - `city`
   - `overflow_len`
   - `has_missing_city`

Дополнительно вычисли `task04_problem_rows` — количество строк, где есть переполнение или отсутствует город.
        

In [ ]:
# YOUR CODE HERE

## Задание 5. Чтение CSV в pandas с преобразованием типов

Прочитай `employees_variant.csv` в `pandas`.

Требования:
1. Разделитель `;`.
2. Поле `hired` преобразуй к дате.
3. Индекс — `employee_id`.
4. Создай столбец `tenure_years` — целое число полных лет стажа на дату `2026-01-01`.
5. Построй `task05_df_filtered` — сотрудники, чья зарплата выше медианы по их департаменту.
6. Контрольная сумма:  
   `task05_checksum = stable_token(task05_df_filtered.reset_index().to_dict(orient="records"))`
        

In [ ]:
# YOUR CODE HERE

## Задание 6. Безопасная запись CSV

Создай файл `employees_export.csv` в каталоге `data`.

Требования:
1. Возьми данные из `task05_df_filtered`.
2. Оставь столбцы `full_name, department, salary`.
3. Добавь столбец `commentary` со строкой вида:  
   `Сотрудник "<имя>" из отдела <department>, salary=<salary>`
4. Запиши CSV так, чтобы кавычки и запятые читались корректно.
5. Сразу перечитай файл и докажи эквивалентность данных:
   `task06_ok = reread_df.equals(expected_df)`
        

In [ ]:
# YOUR CODE HERE

## Задание 7. Многолистовая Excel-книга

Создай файл `analytics_pack.xlsx`.

Требования:
1. На лист `users` запиши `users_df` без индекса.
2. На лист `inventory` запиши содержимое `messy_inventory.csv`.
3. На лист `employees` запиши `employees_df.reset_index()`.
4. После записи перечитай **все листы** через `pd.read_excel(..., sheet_name=None)`.
5. Сохрани словарь количества строк по листам в `task07_sheet_sizes`.
        

In [ ]:
# YOUR CODE HERE

## Задание 8. Форматирование листа employees через openpyxl

Открой `analytics_pack.xlsx` через `openpyxl` и отформатируй лист `employees`.

Требования:
1. `freeze_panes = "B2"`.
2. Заголовок сделать полужирным и залить серым цветом.
3. Для столбца `hired` установить формат `DD.MM.YYYY`.
4. Для столбца `salary` установить ширину не менее 14.
5. Сохрани файл.
6. В `task08_meta` сохрани словарь:
   - `freeze_panes`
   - `salary_width`
   - `header_fill`
        

In [ ]:
# YOUR CODE HERE

## Задание 9. Формулы с относительными и абсолютными ссылками

Добавь в `analytics_pack.xlsx` лист `metrics`.

Требования:
1. В строке 1 размести заголовки: `department, total_salary, avg_salary, max_bonus`.
2. Ниже выпиши уникальные департаменты из `employees_df`.
3. Для каждого департамента запиши формулы Excel:
   - `total_salary` через `SUMIF`
   - `avg_salary` через `AVERAGEIF`
   - `max_bonus` через `MAXIFS`
4. Ссылки на диапазоны в листе `employees` должны быть **абсолютными**.
5. Сохрани список формул первой строки данных в `task09_formulas_preview`.
        

In [ ]:
# YOUR CODE HERE

## Задание 10. Именованный диапазон

В той же книге создай именованный диапазон `salary_band`, который покрывает столбец `salary` на листе `employees` без заголовка.

После этого:
1. На листе `metrics` в ячейку `F1` запиши заголовок `avg_salary_band`.
2. В ячейку `F2` запиши формулу `=AVERAGE(salary_band)`.
3. В переменную `task10_named_range_ref` сохрани текст диапазона.
        

In [ ]:
# YOUR CODE HERE

## Задание 11. Копирование, вставка, вставка столбца и очистка диапазона

Модифицируй лист `inventory` в `analytics_pack.xlsx`.

Требования:
1. После столбца `name` вставь новый столбец `name_len`.
2. Заполни его длиной названия товара.
3. Создай лист `inventory_sample` и скопируй туда заголовок + первые 5 строк данных.
4. На исходном листе `inventory` очисти содержимое диапазона `F3:F4`.
5. Удали полностью последнюю пустую строку, если она появилась.
6. Сохрани итоговый размер листа `inventory` в `task11_shape`.
        

In [ ]:
# YOUR CODE HERE

## Задание 12. Интеграция графика в Excel

На основе листа `metrics` добавь в книгу диаграмму.

Требования:
1. Используй `BarChart` или `LineChart`.
2. По оси категорий — департаменты.
3. В качестве значений — `total_salary`.
4. Размести диаграмму на листе `metrics`, начиная примерно с `H2`.
5. В `task12_chart_title` сохрани заголовок диаграммы.
        

In [ ]:
# YOUR CODE HERE

## Задание 13. Текст xlwings-скрипта для записи DataFrame

Подготовь функцию `task13_xlwings_snippet()`, которая **возвращает строку** с кодом `xlwings`.

Код должен:
1. Открыть `analytics_pack.xlsx`.
2. Получить лист `xlwings_report` или создать его.
3. Записать `dept_summary` в диапазон `B3`.
4. Считать обратно непрерывную таблицу через  
   `.options(pd.DataFrame, expand='table')`.
5. Вернуть считанный объект в переменную `result_df`.

Код возвращается строкой, выполнять его не нужно.
        

In [ ]:
# YOUR CODE HERE

## Задание 14. Имитация поведения `ndim` и `transpose`

Реализуй функцию `normalize_for_excel(data, ndim=None, transpose=False)`.

Функция должна имитировать подготовку полезной нагрузки перед записью в Excel:
- если `transpose=False`, список `[1, 2, 3]` трактуется как строка;
- если `transpose=True`, список `[1, 2, 3]` трактуется как столбец;
- если `ndim=2`, даже одиночное значение должно быть приведено к двумерной структуре.

Подбери 5 тестов и сохрани результаты в словарь `task14_cases`.
        

In [ ]:
# YOUR CODE HERE

## Задание 15. VBA-макрос с `RunPython`

Сформируй строку `task15_vba_module` с кодом VBA-модуля.

Требования:
1. Макрос называется `RefreshDashboard`.
2. Он должен вызывать `RunPython ("import dashboard_tools; dashboard_tools.refresh_dashboard()")`.
3. Добавь короткий комментарий VBA о назначении макроса.
        

In [ ]:
# YOUR CODE HERE

## Задание 16. Модуль xlwings UDF

Подготовь строку `task16_py_module` — текст Python-модуля для `xlwings`.

Требования:
1. Импорт `xlwings as xw`.
2. Функция `weighted_score(values, weights)` помечена `@xw.func`.
3. Аргументы должны быть двумерными: `@xw.arg(..., ndim=2)`.
4. Функция должна возвращать взвешенное среднее по всем элементам.
5. Добавь `@xw.sub`-макрос `write_workbook_name()`, который пишет имя книги в `A1` первого листа.
        

In [ ]:
# YOUR CODE HERE

## Задание 17. Классификация признаков OLTP и OLAP

Прочитай `oltp_olap_statements.csv` и для каждого утверждения присвой класс:
- `OLTP`
- `OLAP`

Сохрани результат в `task17_df`.

После этого вычисли:
`task17_counts = task17_df["class"].value_counts().to_dict()`
        

In [ ]:
# YOUR CODE HERE

## Задание 18. Построение OLAP-подобного куба в pandas

Из `transactions_oltp.csv` построй таблицу-куб.

Требования:
1. Добавь поля:
   - `revenue = units * unit_price * (1 - discount)`
   - `month = период формата YYYY-MM`
2. Построй `task18_cube` как `pivot_table`:
   - индекс: `region, product_group`
   - столбцы: `month`
   - значения: `revenue`
   - агрегирование: `sum`
3. Построй отдельную таблицу `task18_units_cube` для `units`.
4. Найди пару `(region, product_group)` с максимальной общей выручкой и сохрани в `task18_top_slice`.
5. Контрольная сумма:  
   `task18_checksum = stable_token(task18_cube.fillna(0).round(2).reset_index().to_dict(orient="records"))`
        

In [ ]:
# YOUR CODE HERE

## Задание 19. Подготовка простой star-schema

На основе `tx_df` из задания 18 подготовь четыре таблицы:

- `dim_date`
- `dim_product`
- `dim_customer`
- `fact_sales`

Требования:
1. Используй surrogate keys:
   - `date_key`
   - `product_key`
   - `customer_key`
2. В `fact_sales` должны остаться только ключи и числовые показатели:
   - `tx_id, date_key, product_key, customer_key, region, units, unit_price, discount, revenue`
3. Все таблицы сохрани в каталог `data/star_schema/` в CSV.
4. В `task19_shapes` сохрани размеры всех четырёх таблиц.
        

In [ ]:
# YOUR CODE HERE

## Задание 20. Итоговый файл сдачи

Сформируй итоговую книгу `final_submission.xlsx`.

Требования:
1. Листы:
   - `README`
   - `fact_sales`
   - `cube`
   - `insights`
2. На лист `README` запиши:
   - `STUDENT_KEY`
   - `SEED`
   - `task02_checksum`
   - `task05_checksum`
   - `task18_checksum`
3. На лист `fact_sales` запиши `fact_sales`.
4. На лист `cube` запиши `task18_cube` с сохранением индексов.
5. На лист `insights` запиши минимум 3 аналитических вывода, и каждый должен содержать число.
6. В `task20_path` сохрани путь к файлу.
        

In [ ]:
# YOUR CODE HERE